In [ ]:
#Load packages
#based on https://big-fish.readthedocs.io/en/stable/index.html
import os
import nd2reader
import pims
import bigfish
import bigfish.stack
import bigfish.plot
import bigfish.detection
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

In [ ]:
###Select dataset to analyze
input_path = 'D:/Augusto/Images/NSPARC/FISH-Drugs/20250727-NSPARC/TNF-NKF-DMSO' #avoid writing a dash / at the end

input_file = 'HEK293FT-TNF-NKF-DMSO_0002.nd2'

#Define which channel corresponds to fluorescent protein and which one corresponds to nucleus marker ('fp' or 'nucleus')
ch1 = 'nucleus'
ch2 = 'fp'
ch3 = 'cy3'
ch4 = 'cy5'

print("Analysis started")

# print(os.listdir(path))
# datafile = os.path.join(path, file)
# print(datafile)

In [ ]:
###Set output path and file
output_path = 'D:/Augusto/Images/NSPARC/FISH-Jun2025/Analysis/SpotLocation/'

input_date = input_path.split('/')[-2].split('-')[-2]
output_name = input_date + '_' + input_file.split('.')[-2] + "_fish"

output_file = output_path + output_name + '.txt'
output_file_supplement = output_path + output_name + '_supplement.txt'

print('output_file', output_file)
print('output_file_supplement', output_file_supplement)


In [ ]:
#Reads nd2 files from nsparc
#Returns an ND2reader object
def readDataset(path, file):
    
    datafile = os.path.join(path, file)
    print(datafile)
    os.chdir(path)
    print(os.getcwd())
    
     #Load the .nd2 file
    image_raw = pims.open(datafile)
    return image_raw

# image_raw = readDataset(path, file)

# print("Type:", type(image_raw)) 
# print("\nfeatures:\n", image_raw, "\n")

In [ ]:
#Extracts a channel for big fish analysis and returns a np array
#Takes an N2reader object and turns it into an np.ndarray
def getChannel(image_raw, ch):
    
    if ch == ch1:
        print("DAPI analysis")
        channel = 0
    elif ch == ch2:
        print("AF488 analysis")
        channel = 1
    elif ch == ch3:
        print("Cy3 analysis")
        channel = 2
    elif ch == ch4:
        print("Cy5 analysis")
        channel = 3

    number_of_Z = len(image_raw)
    array_list = []
    for z_slice in range(number_of_Z):
        
        frame = image_raw.get_frame_2D(c=channel, t=0, z=z_slice) #iterate t if needed
        array_list.append(frame)
        #print(type(frame))
        
    channel_stack = np.stack(array_list)
    #print(type(channel_stack))
    print(type(channel_stack), channel_stack.shape)
    return (channel_stack)

# cy3_image = getChannel(image_raw, ch3)
# cy5_image = getChannel(image_raw, ch4)

In [ ]:
#Plots all stacks in a np.stack array
def displayNPStack(channel_stack):
    
    #adjust contrast
    vmin = np.percentile(channel_stack, 90)
    vmax = np.percentile(channel_stack, 99.9)
    
    for z_slice in range(len(channel_stack)):
        print("Slice ", z_slice)
        plt.figure(figsize=(10, 10))
        plt.imshow(channel_stack[z_slice], cmap='gray', vmin=vmin, vmax=vmax)
        
        plt.show()
        #Close the figure explicitly to free up memory
        plt.close()
    
    return

#displayNPStack(cy3_image)
    

In [ ]:
#Plots all stacks in a np.stack array using bigfish.plot.plot_yx function
def displayBigFishStack(channel_stack):
    
    for z_slice in range(len(channel_stack)):
        bigfish.plot.plot_yx(channel_stack[z_slice], rescale = False, contrast = True, title = "Cy3 Max Z")
    
    return


In [ ]:
#Read channels in image
image_raw = readDataset(input_path, input_file)
dapi_image = getChannel(image_raw, ch1)
cy3_image = getChannel(image_raw, ch3)
cy5_image = getChannel(image_raw, ch4)


print("Type:", type(image_raw)) 
print("\nfeatures:\n", image_raw, "\n")


In [ ]:
#Plot max Z projections
cy5_image_mZ = bigfish.stack.maximum_projection(cy5_image)
bigfish.plot.plot_yx(cy5_image_mZ, rescale = False, contrast = True, title = "Cy5 Max Z")

cy3_image_mZ = bigfish.stack.maximum_projection(cy3_image)
bigfish.plot.plot_yx(cy3_image_mZ, rescale = False, contrast = True, title = "Cy3 Max Z")

#DAPI Reference
dapi_image_mZ = bigfish.stack.maximum_projection(dapi_image)
bigfish.plot.plot_yx(dapi_image_mZ, rescale = False, contrast = True, title = "DAPI Max Z")


In [ ]:
#Thresholding, turn lowest values into 0

lower_threshold_cy5 = 2
cy5_image_threshold = np.where(cy5_image < lower_threshold_cy5, 0, cy5_image)

lower_threshold_cy3 = 2
cy3_image_threshold = np.where(cy3_image < lower_threshold_cy3, 0, cy3_image)

In [ ]:
#Gaussian Filtering 
gaussian_kernels = [1, 1, 1] #(zyx)

#Cy5 Gaussian
cy5_image_G = bigfish.stack.gaussian_filter(cy5_image_threshold, sigma = gaussian_kernels, allow_negative=False)

cy5_image_G_mZ = bigfish.stack.maximum_projection(cy5_image_G)
bigfish.plot.plot_yx(cy5_image_G_mZ, rescale = False, contrast = True, title = "Cy5_G Max Z")

#Cy3 Gaussian
cy3_image_G = bigfish.stack.gaussian_filter(cy3_image_threshold, sigma = gaussian_kernels, allow_negative=False)

cy3_image_G_mZ = bigfish.stack.maximum_projection(cy3_image_G)
bigfish.plot.plot_yx(cy3_image_G_mZ, rescale = False, contrast = True, title = "Cy3_G Max Z")

In [ ]:
#Detect Gaussian maxima 
#using a multidimensional maximum filter, 
#"a pixel which has the same value in the original and filtered images is a local maximum".
#min_distance = Minimum distance (in pixels) between two spots we want to be able to detect separately.
#https://big-fish.readthedocs.io/en/0.6/detection/spots.html#bigfish.detection.spots_thresholding
min_distance_peaks = 4

cy5_image_G_maxima = bigfish.detection.local_maximum_detection(cy5_image_G, min_distance = min_distance_peaks)
cy3_image_G_maxima = bigfish.detection.local_maximum_detection(cy3_image_G, min_distance = min_distance_peaks)

#Elbow plots to assess brightest peak threshold
# bigfish.plot.plot_elbow(images = cy5_image_G, log_kernel_size = (1, 1, 1), minimum_distance = min_distance_peaks)
# bigfish.plot.plot_elbow(images = cy3_image_G, log_kernel_size = (1, 1, 1), minimum_distance = min_distance_peaks)

In [ ]:
#Threshold of brightest peak after Gaussian filtering

threshold_peak_cy5 = 5
threshold_peak_cy3 = 3

spots_G_cy5, mask_G_cy5 = bigfish.detection.spots_thresholding(image = cy5_image_G, mask_local_max = cy5_image_G_maxima, threshold = threshold_peak_cy5, remove_duplicate=True)
bigfish.plot.plot_detection(cy5_image_G_mZ, spots_G_cy5, contrast=True)

spots_G_cy3, mask_G_cy3 = bigfish.detection.spots_thresholding(image = cy3_image_G, mask_local_max = cy3_image_G_maxima, threshold = threshold_peak_cy3, remove_duplicate=True)
bigfish.plot.plot_detection(cy3_image_G_mZ, spots_G_cy3, contrast=True)


In [ ]:
#Colocalization of spots detected in cy5 and cy3 channels

distance_threshold_nm = 300 #maximum distance between spots in nm
voxel_size_nm = [195, 73, 73] #z, y, x size in nm

spots_col_G_cy5, spots_col_G_cy3, distances_G = bigfish.multistack.detect_spots_colocalization(spots_1 = spots_G_cy5, spots_2 = spots_G_cy3, voxel_size = voxel_size_nm, threshold = distance_threshold_nm,
                                               return_indices = False, return_threshold=False)

bigfish.plot.plot_detection(cy5_image_G_mZ, spots_col_G_cy5, contrast=True)

bigfish.plot.plot_detection(cy3_image_G_mZ, spots_col_G_cy3, contrast=True)

In [ ]:
#Define the middle position (x, y, z) between spots detected in cy5 and cy3 channels

spots_col_G_cy5cy3 = np.round(np.mean([spots_col_G_cy5, spots_col_G_cy3], axis=0)).astype(int)

detected_pairs = [] 
for row_cy5, row_cy3, row_cy5cy3 in zip(spots_col_G_cy5, spots_col_G_cy3, spots_col_G_cy5cy3):
        
    detected_pairs.append([row_cy5[0], row_cy5[1], row_cy5[2], row_cy3[0], row_cy3[1], row_cy3[2], 
                         row_cy5cy3[0], row_cy5cy3[1], row_cy5cy3[2]])
            
print(len(detected_pairs), "FISH-spot pairs were detected")
        

In [ ]:
#Supervise selected pairs of FISH spots
output_data = []

#Plot the center images of detected spots
vmin_nuc = np.percentile(dapi_image, 40) #set contrast
vmax_nuc = np.percentile(dapi_image, 100) #set contrast

vmin_cy3 = np.percentile(cy3_image, 60) #set contrast
vmax_cy3 = np.percentile(cy3_image, 99.99) #set contrast

vmin_cy5 = np.percentile(cy5_image, 60) #set contrast
vmax_cy5 = np.percentile(cy5_image, 99.99) #set contrast

for sp in detected_pairs:
    
    sp_z = sp[6]
    sp_y = sp[7]
    sp_x = sp[8]
    #print(sp_z, sp_x, sp_y)
    
    x_low = sp_x - 100
    x_high = sp_x + 100
    y_low = sp_y - 100
    y_high = sp_y + 100
    y_arrow = sp_y - 12

    x_low = np.maximum(x_low, 0)
    x_high = np.minimum(x_high, 2047)
    y_low = np.maximum(y_low, 0)
    y_high = np.minimum(y_high, 2047)
    y_arrow = np.maximum(y_arrow, 0)
        
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw={'width_ratios': [1, 1, 1]})
    
    ax1.scatter(sp_x, y_arrow, marker='v', color = "b", linewidth=5, alpha = 1, s = 100)
    ax1.imshow(dapi_image[sp_z], origin='upper', cmap = 'gray', vmin = vmin_nuc, vmax = vmax_nuc)
    ax1.set_xlim(x_low, x_high) #zoom to mask currently analyzed
    ax1.set_ylim(y_low, y_high)
    ax1.set_ylim(ax1.get_ylim()[::-1])  # Reverse the y-axis limits
    ax1.set_title("DAPI-average in\n" + "z:" + str(sp_z) + " y:" + str(sp_y) + " x:" + str(sp_x))
    
    ax2.scatter(sp_x, y_arrow, marker='v', color = "y", linewidth=5, alpha = 1, s = 100)
    ax2.imshow(cy3_image[sp_z], origin='upper', cmap = 'gray', vmin = vmin_cy3, vmax = vmax_cy3)
    ax2.set_xlim(x_low, x_high) #zoom to mask currently analyzed
    ax2.set_ylim(y_low, y_high)
    ax2.set_ylim(ax2.get_ylim()[::-1])  # Reverse the y-axis limits
    ax2.set_title("Cy3-center in\n" + "z:" + str(sp[3]) + " y:" + str(sp[4]) + " x:" + str(sp[5]))
    
    ax3.scatter(sp_x, y_arrow, marker='v', color = "r", linewidth=5, alpha = 1, s = 100)
    ax3.imshow(cy5_image[sp_z], origin='upper', cmap = 'gray', vmin = vmin_cy5, vmax = vmax_cy5)
    ax3.set_xlim(x_low, x_high) #zoom to mask currently analyzed
    ax3.set_ylim(y_low, y_high)
    ax3.set_ylim(ax3.get_ylim()[::-1])  # Reverse the y-axis limits
    ax3.set_title("Cy5-center in\n" + "z:" + str(sp[0]) + " y:" + str(sp[1]) + " x:" + str(sp[2]))
    
    plt.show()
    
    ask_FISH = "Is this pair of spots good for further analysis (y/n)\n"
    answer_FISH = input(ask_FISH)
        
    if answer_FISH == 'y':
        output_data.append(sp)
        
print("analysis finished")


In [ ]:
#convert selected_data toy pandas data frame and save as a .txt file

headers = [
    'cy5_z', 'cy5_y', 'cy5_x',
    'cy3_z', 'cy3_y', 'cy3_x',
    'cy5cy3_z', 'cy5cy3_y', 'cy5cy3_x'
]

print(len(output_data), "out of", len(detected_pairs), "FISH-spot pairs were selected\n")

output_df = pd.DataFrame(output_data, columns=headers)
print(output_df, "\n")
        
# Save to file
output_df.to_csv(output_file, sep='\t', index=True, index_label="id")
print('data saved in:', output_file)

In [ ]:
## Generate a supplement file with the parameters used in the current analysis

current_date = datetime.now() #Obtain date
formatted_date = current_date.strftime("%Y-%m-%d")

formatted_date_parameters = "Analysis date: "  + formatted_date + "\n"
input_date_parameters = "input_date: " + str(input_path.split('/')[-2]) + "\n"
input_file_parameters = "input_file: " + str(input_file) + "\n"
lower_threshold_cy5_parameters = "lower_threshold_cy5: " + str(lower_threshold_cy5) + "\n"
lower_threshold_cy3_parameters = "lower_threshold_cy3: " + str(lower_threshold_cy3) + "\n"
gaussian_kernels_parameters = "gaussian_kernels: " + str(gaussian_kernels) + "\n"
min_distance_peaks_parameters = "min_distance_peaks: " + str(min_distance_peaks) + "\n"
threshold_peak_cy5_parameters = "threshold_peak_cy5: " + str(threshold_peak_cy5) + "\n"
threshold_peak_cy3_parameters = "threshold_peak_cy3: " + str(threshold_peak_cy3) + "\n"
distance_threshold_nm_parameters = "distance_threshold_nm: " + str(distance_threshold_nm) + "\n"
voxel_size_nm_parameters = "voxel_size_nm: " + str(voxel_size_nm) + "\n"
pairs_detected_parameters = "FISH-spot pairs detected: " + str(len(detected_pairs)) + "\n"
pairs_selected_parameters = "FISH-spot pairs selected: " + str(len(output_data)) + "\n"

print(formatted_date_parameters)
print(input_date_parameters)
print(input_file_parameters)
print(lower_threshold_cy5_parameters)
print(lower_threshold_cy3_parameters)
print(gaussian_kernels_parameters)
print(min_distance_peaks_parameters)
print(threshold_peak_cy5_parameters)
print(threshold_peak_cy3_parameters)
print(distance_threshold_nm_parameters)
print(voxel_size_nm_parameters)
print(pairs_detected_parameters)
print(pairs_selected_parameters)

# if not os.path.exists(output_folder): # Check if the folder exists
#     #Create the folder
#     os.makedirs(output_folder)
#     print("Folder created successfully!")
# else:
#     print("Folder already exists.")

try:
    with open(output_file_supplement, 'w') as f:
        
        f.write(formatted_date_parameters)
        f.write(input_date_parameters)
        f.write(input_file_parameters)
        f.write(lower_threshold_cy5_parameters)
        f.write(lower_threshold_cy3_parameters)
        f.write(gaussian_kernels_parameters)
        f.write(min_distance_peaks_parameters)
        f.write(threshold_peak_cy5_parameters)
        f.write(threshold_peak_cy3_parameters)
        f.write(distance_threshold_nm_parameters)
        f.write(voxel_size_nm_parameters)
        f.write(pairs_detected_parameters)
        f.write(pairs_selected_parameters)
                
except IOError:
    print("Could not open file {output_file_supplement} for writing") #Handle the case where the file cannot be opened or written to

print("File:\n", output_file_supplement, "\ngenerated")
